<a href="https://colab.research.google.com/github/NimethMethnukaKK/CNN/blob/Nimeth_220394F/Jute_Pest_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import Model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import os
import requests
import zipfile
from io import BytesIO

In [8]:
!pip install keras numpy pandas matplotlib scikit-learn

In [9]:
!pip install ucimlrepo


# **1. CNN for image classification**

In [10]:
# URL of the dataset ZIP file (replace with the actual Jute Pest dataset URL)
url = "https://archive.ics.uci.edu/static/public/920/jute+pest+dataset.zip"

# Download and extract
response = requests.get(url)
with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
    zip_ref.extractall("jute_dataset")

print("Dataset downloaded and extracted!")


Dataset downloaded and extracted!


In [11]:
data_dir = "jute_dataset"
for root, dirs, files in os.walk(data_dir):
    print(root, len(files))

jute_dataset 1


In [12]:
print(os.listdir("jute_dataset"))

['Jute_Pest_Dataset.zip']


In [13]:
inner_zip = "jute_dataset/Jute_Pest_Dataset.zip"

with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
    zip_ref.extractall("jute_dataset")

In [26]:
data_dir = "jute_dataset/Jute_Pest_Dataset/train"  # extracted folder

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.3
)

train_gen = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=16,
    subset='training',
    seed=42
)

val_gen = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=32,
    subset='validation',
    seed=42
)



Found 4517 images belonging to 17 classes.
Found 1926 images belonging to 17 classes.


In [27]:
from tensorflow.keras import regularizers

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 17)             │         4,369 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,521,297 (24.88 MB)

 Trainable params: 6,520,849 (24.88 MB)

 Non-trainable params: 448 (1.75 KB)

In [28]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

opt = Adam(learning_rate=0.0005)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,     # reduce by half
    patience=2,     # if val_loss not improving for 2 epochs
    min_lr=1e-6,
    verbose=1
)

model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=[early_stop, reduce_lr]
)

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()


Epoch 1/30
283/283 ━━━━━━━━━━━━━━━━━━━━ 303s 1s/step - accuracy: 0.1095 - loss: 5.7091 - val_accuracy: 0.0659 - val_loss: 6.2479 - learning_rate: 5.0000e-04
Epoch 2/30
283/283 ━━━━━━━━━━━━━━━━━━━━ 301s 1s/step - accuracy: 0.1339 - loss: 3.4070 - val_accuracy: 0.0774 - val_loss: 3.5030 - learning_rate: 5.0000e-04
Epoch 3/30
283/283 ━━━━━━━━━━━━━━━━━━━━ 305s 1s/step - accuracy: 0.1295 - loss: 3.3649 - val_accuracy: 0.1251 - val_loss: 3.3814 - learning_rate: 5.0000e-04
Epoch 4/30
283/283 ━━━━━━━━━━━━━━━━━━━━ 319s 1s/step - accuracy: 0.1363 - loss: 3.3067 - val_accuracy: 0.1205 - val_loss: 3.2784 - learning_rate: 5.0000e-04
Epoch 5/30
283/283 ━━━━━━━━━━━━━━━━━━━━ 298s 1s/step - accuracy: 0.1320 - loss: 3.2347 - val_accuracy: 0.1121 - val_loss: 3.2966 - learning_rate: 5.0000e-04
Epoch 6/30
283/283 ━━━━━━━━━━━━━━━━━━━━ 0s 928ms/step - accuracy: 0.1265 - loss: 3.2316
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
283/283 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accurac

KeyboardInterrupt: 

In [17]:
y_true = val_test_gen.classes
y_pred = np.argmax(model.predict(val_test_gen), axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 263ms/step
[[ 1  0  2  4 11  4  1  1  5  6  5  7  3  3  3  2  1]
 [ 0  0  2  9 11  3 11  1  0 10  5  9  5  3 10  8  5]
 [ 5  1  0  4 17 10  9  5  7 13  4 18  5  9  6  1  5]
 [ 1  3  1  7 21 10 10  4  1 11  3 13  6  5 10  7  4]
 [ 1  2  2 15 15  8 23  8  1 22  5 24  3  6  6  5 13]
 [ 4  0  2 12 15  9  6  6  9 29  4  7  6  5  2  4  4]
 [ 0  3  2 13 16  7 17  4  4 16  5 12  7  5  8  8 10]
 [ 0  0  1  7 23  0  7  2  5 20  4 12  5  8  6  3  8]
 [ 2  2  2  4  8  7  5  1  2 13  5  9  4  5  6  7  2]
 [ 4  4  6 17 38  8 17  6  4 25  7 18  7 10  9 13  9]
 [ 0  1  1  6 16  5  9  7  0 15  5  4  1  8  5  1  9]
 [ 1  5  5  5 19  5 16  4  4 18  3 11  8  5  9  8 13]
 [ 2  2  0  5 25  9  7  2  3 18  4  8  3  4 11  8  4]
 [ 3  1  1  4 16  5  4  7  1 16  2  7  3  4  8  7  7]
 [ 0  1  3  3 14  4  5  3  3 19  4  7  6  1  3  5  6]
 [ 1  2  2  5 16  4 12  4  0  9  4  9  1  4  7  4  6]
 [ 0  2  4  4  9  7 12  2  3 21  4  5  3  8  7  3  8]]
              precision    recall  f1-s

# **2. Comparing network with state-of-the-art networks**



In [18]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128,128,3))
for layer in base_model.layers[:-4]:
    layer.trainable = False

x = layers.Flatten()(base_model.output)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_gen.num_classes, activation='softmax')(x)

vgg_model = Model(base_model.input, output)

vgg_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

vgg_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,765,585 (60.14 MB)

 Trainable params: 8,130,321 (31.01 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [19]:
vgg_history = vgg_model.fit(
    train_gen,
    validation_data=val_test_gen,
    epochs=15
)

Epoch 1/15
 11/142 ━━━━━━━━━━━━━━━━━━━━ 16:36 8s/step - accuracy: 0.0825 - loss: 3.2076

KeyboardInterrupt: 